# 12 — Agentic RAG

Module 11 retrieved once and generated. That is enough when one file holds the answer.

Real questions often do not. "The same title arrived damaged twice — how long until the refund hits the card?" is two facts: *what we do the second time*, and *how long a refund takes*. Those live in two files. One nearest-neighbour grab will pick one of them.

Today the retrieve is a **tool**. The official loop from 03 decides whether to search again, with a rewritten query, or to stop. That is agentic RAG. The store did not change. Chroma is still Chroma.
**Module 11 — one retrieve, one generate**

```mermaid
flowchart LR
    A["question"] --> B["retrieve k=3"]
    B --> C["generate"]
```

**Module 12 — retrieve is a tool, and the loop may call it again**

```mermaid
graph TD
    A[Question] --> B[Retrieve documents]

    B --> C{Enough context}

    C -->|Yes| D[Generate answer]

    C -->|Missing fact| E[Rewrite query]
    E --> B

    C -->|Not in corpus| F[Say not found]
    F --> G[END]
```


## 1. Learn

```
11  one retrieve, one generate
12  you are here — retrieve is a tool; the loop may call it twice
14  a retrieved file can carry instructions
```

Single-shot RAG fails on at least three shapes:

| Shape | Why one retrieve is not enough |
|---|---|
| Multi-hop | The second fact is not in the first file. |
| Bad phrasing | The user said "SLA"; the file said "first reply." |
| Absent | Nearest is still a file. The generate step has to refuse. |

The agentic move is not a new store. It is the loop you already wrote, with `retrieve(query)` as the only tool.

```
Thought (inside the model)
   |
   +-- retrieve("damaged twice")
   |         observation: policy_damaged_media.md  (refund, no third copy)
   |
   +-- retrieve("refund how many days after approval")
   |         observation: policy_refunds.md  (5 to 10 business days)
   |
   +-- stop: both facts are in the list
```

Cap the turns. An unanswerable question will otherwise search forever.

We keep **one file = one chunk** and **one store = Chroma**. LlamaIndex would wrap this loop. We will not install it. Module 11 already made that point.


## 2. Do

### Same folder, same embeddings, in this kernel

No import from module 11. Rebuild the collection here so this notebook stands alone.


In [1]:
from pathlib import Path
import json
import os

from chromadb import Client
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI


load_dotenv(find_dotenv(usecwd=True))
ROOT = Path(find_dotenv(usecwd=True)).parent

api_key = os.environ.get("OPENAI_API_KEY", "").strip()
model = os.environ.get("MODEL_DEFAULT", "").strip()
embed_model = os.environ.get("EMBEDDING_MODEL", "").strip()
assert api_key, "OPENAI_API_KEY is missing."
assert model, "MODEL_DEFAULT is missing from .env."
assert embed_model, "EMBEDDING_MODEL is missing. Copy the line from .env.example."

client = OpenAI()
CORPUS = ROOT / "data" / "corpus"
files = sorted(CORPUS.glob("*.md"))


def embed(text: str) -> list[float]:
    return client.embeddings.create(model=embed_model, input=text).data[0].embedding


ids, documents, metadatas, vectors = [], [], [], []
for path in files:
    text = path.read_text()
    ids.append(path.name)
    documents.append(text)
    metadatas.append({"path": path.name})
    vectors.append(embed(text))

chroma = Client()
collection = chroma.create_collection("corpus")
collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=vectors)
print("OPENAI_API_KEY is set:", True)
print("MODEL_DEFAULT:", model)
print("stored:", collection.count())


OPENAI_API_KEY is set: True
MODEL_DEFAULT: gpt-5.4-nano
stored: 36


### `retrieve` is an ordinary function

Print `k=2` once so you see **both** files exist. The tool later uses `k=1`: one call cannot carry both facts.


In [2]:
def retrieve(query: str, k: int = 2):
    got = collection.query(query_embeddings=[embed(query)], n_results=k)
    rows = []
    for i in range(len(got["ids"][0])):
        rows.append(
            {
                "id": got["ids"][0][i],
                "distance": got["distances"][0][i],
                "document": got["documents"][0][i],
            }
        )
    return rows


QUESTION = (
    "If the same title arrives damaged twice, "
    "how long after approval does the refund take?"
)
for row in retrieve(QUESTION):
    print(f"{row['distance']:.3f}  {row['id']}")
    print(row["document"].splitlines()[0])
    print()


0.895  policy_damaged_media.md
# Damaged media

0.937  policy_refunds.md
# Refunds



Two files. `policy_damaged_media.md` is the second-time rule (refund, no third copy). `policy_refunds.md` is the clock: **5 to 10 business days**. The tool will not return both at once.

### Single-shot, `k=1`

Same generate as 11. One file only. The refund clock should be missing, or invented.


In [3]:
def generate(question: str, rows):
    packed = "\n\n".join(f"# {row['id']}\n{row['document']}" for row in rows)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer using only the documents. "
                    "If they do not contain the answer, say you do not know. "
                    "Do not invent a number, a date, or a phone number."
                ),
            },
            {
                "role": "user",
                "content": "Documents:\n\n" + packed + "\n\nQuestion: " + question,
            },
        ],
        max_completion_tokens=160,
        reasoning_effort="none",
    )
    return response.choices[0].message.content, response.usage.prompt_tokens


one = retrieve(QUESTION, k=1)
text_one, tokens_one = generate(QUESTION, one)
print("used:", [row["id"] for row in one])
print("prompt_tokens:", tokens_one)
print()
print(text_one)


used: ['policy_damaged_media.md']
prompt_tokens: 133

I don’t know. The provided document only says: “If the same title arrives damaged twice, offer a refund instead of a third copy,” but it does not state how long the refund takes after approval.


Cover that sentence. If it said "I do not know" how long the refund takes, single-shot was honest and incomplete. If it invented a number of days, the store did not save you.

### Retrieve as a tool

The official loop from 03. One tool. The model rewrites the query; we run `retrieve`; we send the files back as a `tool` message.


In [4]:
def schema(tool_name, description, **props):
    return {"type": "function", "function": {
        "name": tool_name, "description": description,
        "parameters": {"type": "object", "properties": props, "required": list(props)}}}


tools = [
    schema(
        "retrieve",
        "Search the shop policy and ticket files. "
        "Argument is a short search query, not the user's whole question. "
        "Returns the single nearest file. If you need a second fact, call again with a different query.",
        query={"type": "string"},
    ),
]


def run_retrieve(call):
    args = json.loads(call.function.arguments or "{}")
    query = args.get("query", "")
    rows = retrieve(query, k=1)
    names = [row["id"] for row in rows]
    packed = []
    for row in rows:
        packed.append(f"{row['id']} (distance {row['distance']:.3f})\n{row['document']}")
    return query, names, "\n\n".join(packed)


print([t["function"]["name"] for t in tools])


['retrieve']


### The loop

Cap of 5. Print every query the model asked for. If `finish_reason` is `stop`, we have a sentence. An unanswerable question should stop without a sixth retrieve.


In [5]:
SYSTEM = (
    "You answer from the shop files. Use retrieve. "
    "If the first files are missing a fact, call retrieve again with a different query. "
    "If the files do not contain the answer, say you do not know. "
    "Do not invent a number, a date, or a phone number."
)


def run_loop(question, max_turns=5):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    queries = []
    files_used = []
    final_text = None
    prompt_tokens = 0
    for turn in range(max_turns):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            max_completion_tokens=200,
            reasoning_effort="none",
        )
        prompt_tokens = prompt_tokens + response.usage.prompt_tokens
        message = response.choices[0].message
        print("--- turn", turn + 1, "finish_reason:", response.choices[0].finish_reason, "---")
        if not message.tool_calls:
            final_text = message.content
            print(final_text)
            break
        messages.append(message)
        for call in message.tool_calls:
            query, names, result = run_retrieve(call)
            queries.append(query)
            files_used.extend(names)
            print("retrieve:", query)
            print("  files:", names)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})
    else:
        print("stopped: cap was", max_turns)
    return {
        "queries": queries,
        "files_used": files_used,
        "n_retrieves": len(queries),
        "final_text": final_text,
        "prompt_tokens": prompt_tokens,
    }


looped = run_loop(QUESTION)
print()
print("n_retrieves:", looped["n_retrieves"])
print("queries:    ", looped["queries"])


--- turn 1 finish_reason: tool_calls ---
retrieve: refund takes how long after approval same title arrives damaged twice
  files: ['policy_damaged_media.md']


--- turn 2 finish_reason: tool_calls ---


retrieve: refund takes how long after approval
  files: ['policy_refunds.md']


--- turn 3 finish_reason: stop ---
If the same title arrives damaged twice, you can request a refund (instead of a third replacement). Refunds take **5 to 10 business days after support approves the request**.

n_retrieves: 2
queries:     ['refund takes how long after approval same title arrives damaged twice', 'refund takes how long after approval']


The second query should mention refund or days. `policy_refunds.md` should appear. The sentence should now have **5 to 10 business days**, and no third replacement.

If nano stopped after one retrieve and still said it did not know the clock, that is the same honest miss as single-shot. Rerun the cell, or set `MODEL_STRONG` in `.env` for this notebook only.

### An unanswerable, same loop

The corpus has no CEO mobile number. The loop should retrieve, see that nothing helps, and stop. Cap of 5 is the backstop.


In [6]:
absent = run_loop("What is the CEO's personal mobile number?")
print()
print("n_retrieves:", absent["n_retrieves"])
print("queries:    ", absent["queries"])


--- turn 1 finish_reason: tool_calls ---


retrieve: CEO personal mobile number
  files: ['policy_escalations.md']


--- turn 2 finish_reason: stop ---
I don’t know. Our shop files don’t include (or publish) the CEO’s personal mobile number, and I can’t invent one.

n_retrieves: 1
queries:     ['CEO personal mobile number']


## 3. Observe

Same two questions, two methods. The store did not change. The loop did.


In [7]:
print(f"{'method':<16} {'n_retrieves':>11}  {'top / files'}")
print(f"{'single-shot k=1':<16} {1:11}  {[row['id'] for row in one]}")
print(f"{'loop, damaged':<16} {looped['n_retrieves']:11}  {looped['files_used']}")
print(f"{'loop, CEO':<16} {absent['n_retrieves']:11}  {absent['files_used']}")
print()
print("single-shot tokens:", tokens_one)
print("loop tokens:       ", looped["prompt_tokens"])
print("CEO loop tokens:   ", absent["prompt_tokens"])


method           n_retrieves  top / files
single-shot k=1            1  ['policy_damaged_media.md']
loop, damaged              2  ['policy_damaged_media.md', 'policy_refunds.md']
loop, CEO                  1  ['policy_escalations.md']

single-shot tokens: 133
loop tokens:        1050
CEO loop tokens:    566


Things to notice:

- `retrieve`'s `query` is a **string the model wrote**. That is module 02 again. You can refuse a query. You can log it.
- Two retrieves cost more than one. The extra spend is the point of 05, applied to RAG. Do not loop when one file is enough.
- The CEO question still returned files. Nearest is not relevant. The generate step is what refused — or failed to.
- Chroma did not grow a planner. The official loop did.

Module 14 will put a hostile sentence in one of those files. The loop you just wrote will fetch it the same way it fetched `policy_refunds.md`.

## 4. Challenge

Official loop, same `retrieve` tool. A new two-hop question:

> Helena received the wrong album. Who is her assigned support representative, and how soon should support send a first reply?

The ticket names the representative. The first-reply SLA is in the support-hours policy. One retrieve will miss one of them.

Bind:

- `n_retrieves` — how many times you called `retrieve`
- `files_used` — the file names that came back
- `final_text` — the last sentence

The next cell checks that you retrieved at least twice, that **Steve** appears, and that the sentence mentions a **business** day. It does not score the wording.


In [ ]:
# n_retrieves, files_used, final_text = ...


In [ ]:
assert n_retrieves >= 2, "this question needs a second retrieve"
assert files_used, "files_used should be the names retrieve returned"
assert final_text and str(final_text).strip(), "final_text should be the last sentence"
assert "steve" in str(final_text).lower(), "Helena's assigned rep is Steve Johnson"
assert "business" in str(final_text).lower(), "first reply is within 1 business day"
print("looks good")
